<div style="background:linear-gradient(135deg,#4c0519 0%,#be123c 55%,#fb7185 100%);border-radius:18px;padding:30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#fecdd3;font-weight:700;text-transform:uppercase">Chapter 62 · Solutions</div>
  <div style="font-size:30px;font-weight:900;margin:8px 0 4px">Probability Sampling — Worked Solutions ✅</div>
  <div style="font-size:14px;color:#fff1f2;max-width:720px;line-height:1.6">Five challenges, each verified in code.</div>
</div>

## ⚙️ Setup &amp; population

In [1]:
import numpy as np, pandas as pd
rng = np.random.default_rng(620)
# two strata with very different means
A = rng.normal(40, 6, 60000); B = rng.normal(80, 6, 40000)
pop = np.concatenate([A,B]); strat = np.array(["A"]*60000+["B"]*40000)
MU = pop.mean(); print(f"true mean = {MU:.2f}")

true mean = 55.99


<div style="background:#fff1f2;border-left:5px solid #e11d48;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#be123c;letter-spacing:1px">CHALLENGE 1</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">SRS is unbiased</div>
<div style="color:#4a5578;margin-top:5px">Draw 3,000 simple random samples of size 200 and confirm the average estimate equals the true mean.</div>
</div>

In [2]:
means=np.array([pop[rng.choice(len(pop),200,replace=False)].mean() for _ in range(3000)])
print(f"average of SRS estimates = {means.mean():.2f}  (true {MU:.2f})")
print(f"SRS standard error = {means.std():.3f}")

average of SRS estimates = 55.96  (true 55.99)
SRS standard error = 1.442


<div style="background:#fff1f2;border-left:5px solid #e11d48;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#be123c;letter-spacing:1px">CHALLENGE 2</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">Stratified beats SRS</div>
<div style="color:#4a5578;margin-top:5px">Draw stratified samples (proportional allocation, n=200) and compare the standard error to SRS.</div>
</div>

In [3]:
idxA=np.where(strat=="A")[0]; idxB=np.where(strat=="B")[0]
def stratified(n=200):
    a=pop[rng.choice(idxA, round(n*0.6), replace=False)]
    b=pop[rng.choice(idxB, round(n*0.4), replace=False)]
    return np.concatenate([a,b]).mean()
sm=np.array([stratified() for _ in range(3000)])
srs=np.array([pop[rng.choice(len(pop),200,replace=False)].mean() for _ in range(3000)])
print(f"stratified SE = {sm.std():.3f}")
print(f"SRS SE        = {srs.std():.3f}")
print(f"variance cut  = {(1-(sm.std()/srs.std())**2)*100:.0f}%")

stratified SE = 0.428
SRS SE        = 1.449
variance cut  = 91%


<div style="background:#fff1f2;border-left:5px solid #e11d48;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#be123c;letter-spacing:1px">CHALLENGE 3</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">The cluster penalty</div>
<div style="color:#4a5578;margin-top:5px">Split the sorted population into homogeneous clusters of 100 and sample 2 clusters. Show the SE is larger than SRS at the same n.</div>
</div>

In [4]:
order=np.argsort(pop); clustered=pop[order]; cl=np.arange(len(pop))//100
def cluster(n_cl=2):
    chosen=rng.choice(cl.max()+1, n_cl, replace=False)
    return clustered[np.isin(cl, chosen)].mean()
cm=np.array([cluster() for _ in range(3000)])
print(f"cluster SE (2x100 = n=200) = {cm.std():.2f}")
print(f"SRS SE     (n=200)         = {srs.std():.2f}")
print("clusters of similar units carry less information -> higher variance")

cluster SE (2x100 = n=200) = 14.33
SRS SE     (n=200)         = 1.45
clusters of similar units carry less information -> higher variance


<div style="background:#fff1f2;border-left:5px solid #e11d48;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#be123c;letter-spacing:1px">CHALLENGE 4</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">The periodicity trap</div>
<div style="color:#4a5578;margin-top:5px">Build a list whose values repeat with period k and show systematic sampling (every k-th) returns a biased estimate.</div>
</div>

In [5]:
k=10; cycle=np.tile([10,20,30,40,50,60,70,80,90,100], 1000)  # period 10
print(f"full-list mean = {cycle.mean():.1f}")
for start in [0, 3, 7]:
    est=cycle[start::k].mean()
    print(f"every {k}-th from start {start}: estimate = {est:.1f}  (off by {est-cycle.mean():+.1f})")
print("each systematic pass hits ONE repeated value -> wildly biased")

full-list mean = 55.0
every 10-th from start 0: estimate = 10.0  (off by -45.0)
every 10-th from start 3: estimate = 40.0  (off by -15.0)
every 10-th from start 7: estimate = 80.0  (off by +25.0)
each systematic pass hits ONE repeated value -> wildly biased


<div style="background:#fff1f2;border-left:5px solid #e11d48;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#be123c;letter-spacing:1px">CHALLENGE 5</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">Proportional allocation</div>
<div style="color:#4a5578;margin-top:5px">Verify that proportional allocation samples each stratum in proportion to its share of the population.</div>
</div>

In [6]:
shareA=len(idxA)/len(pop); n=500
print(f"stratum A is {shareA*100:.0f}% of the population")
print(f"proportional allocation draws {round(n*shareA)} of {n} from A  ({round(n*shareA)/n*100:.0f}%)")
print("the sample mirrors the population composition by construction")

stratum A is 60% of the population
proportional allocation draws 300 of 500 from A  (60%)
the sample mirrors the population composition by construction


---
<div style="text-align:center;color:#8b94b3;font-size:12px;margin-top:10px">Statistics, Data Science and AI: A Visual Handbook · © 2026 John Fisher</div>